## Module 3 - Global Ranking Score, Research Impact Score, Faculty-Student Ratio

In [14]:
# Module 3 - Safety check: make sure university_cleaned exists
# (in case this cell is run without first running Modules 1-2 above)
try:
    university_cleaned
except NameError:
    import pandas as pd
    import numpy as np
    import os
    print("'university_cleaned' not found in memory — loading from CSV instead.")
    university_cleaned = pd.read_csv("data/university_cleaned.csv")

print("university_cleaned shape:", university_cleaned.shape)
university_cleaned.head()

university_cleaned shape: (3692, 18)


,source,year,institution,country,region,rank_numeric,rank_display,previous_rank,overall_score,academic_reputation,employer_reputation,faculty_student_ratio_score,citations_per_faculty,international_faculty_score,international_students_score,international_research_network,employment_outcomes,sustainability_score
0,QS,2026,Massachusetts Institute of Technology (MIT),United States,Americas,1.0,1,1,100.0,100.0,100.0,100.0,100.0,100.0,91.6,94.1,100.0,93.8
1,QS,2026,Imperial College London,United Kingdom,Europe,2.0,2,2,99.4,99.6,100.0,99.3,95.0,100.0,100.0,97.5,95.9,98.3
2,QS,2026,Stanford University,United States,Americas,3.0,3,6,98.9,100.0,100.0,100.0,99.7,94.2,73.5,96.5,100.0,95.4
3,QS,2026,University of Oxford,United Kingdom,Europe,4.0,4,3,97.9,100.0,100.0,100.0,91.0,98.8,98.6,100.0,100.0,77.9
4,QS,2026,Harvard University,United States,Americas,5.0,5,4,97.7,100.0,100.0,98.3,100.0,79.1,81.4,99.4,100.0,77.8


In [15]:
# Module 3 - Faculty-Student Ratio (standardized, comparable across sources)
# QS 'faculty_student_ratio_score' is already a 0-100 score where HIGHER = better.
# THE 'faculty_student_ratio_score' is the raw students-to-staff ratio
# (e.g. 12.3 students per staff member), where LOWER = better.
# To combine them into one comparable metric, we min-max scale each source
# separately to 0-100, inverting THE's raw ratio so higher always means better.

def minmax_0_100(s):
    lo, hi = s.min(), s.max()
    if pd.isna(lo) or pd.isna(hi) or hi == lo:
        return pd.Series(np.nan, index=s.index)
    return 100 * (s - lo) / (hi - lo)

fsr = university_cleaned['faculty_student_ratio_score']
src = university_cleaned['source']

fsr_standardized = pd.Series(np.nan, index=university_cleaned.index)

qs_mask = src == 'QS'
fsr_standardized.loc[qs_mask] = minmax_0_100(fsr.loc[qs_mask])          # already higher=better

the_mask = src == 'THE'
the_scaled = minmax_0_100(fsr.loc[the_mask])
fsr_standardized.loc[the_mask] = 100 - the_scaled                       # invert: lower ratio -> higher score

university_cleaned['faculty_student_ratio_std'] = fsr_standardized.round(2)
university_cleaned[['source', 'institution', 'faculty_student_ratio_score', 'faculty_student_ratio_std']].sample(10, random_state=1)

,source,institution,faculty_student_ratio_score,faculty_student_ratio_std
1672,THE,Sapienza University of Rome,27.3,88.20
2642,THE,New Valley University,8.6,96.50
1237,QS,Instituto Tecnológico de Santo Domingo (INTEC),36.2,35.56
2374,THE,The Catholic University of America,10.8,95.52
1163,QS,University of Louisville,43.8,43.23
1422,QS,Harper Adams University,7.1,6.16
994,QS,University of Bahrain,8.1,7.17
2999,THE,Yokohama National University,13.2,94.45
1402,QS,Western Michigan University,15.7,14.85
1578,THE,University of North Carolina at Chapel Hill,8.0,96.76


In [16]:
# Module 3 - Research Impact Score
# Built from the two research-related metrics available in both sources:
# 'citations_per_faculty' (research quality/citation strength) and
# 'international_research_network' (breadth of research collaboration).
# Both are already higher=better, but each source uses its own scale,
# so we min-max scale each column within its own source before averaging.

research_cols = ['citations_per_faculty', 'international_research_network']

research_std = pd.DataFrame(index=university_cleaned.index)
for col in research_cols:
    standardized = pd.Series(np.nan, index=university_cleaned.index)
    for s in ['QS', 'THE']:
        mask = src == s
        standardized.loc[mask] = minmax_0_100(university_cleaned.loc[mask, col])
    research_std[col] = standardized

university_cleaned['research_impact_score'] = research_std.mean(axis=1, skipna=True).round(2)
university_cleaned[['source', 'institution', 'citations_per_faculty', 'international_research_network', 'research_impact_score']].sample(10, random_state=1)

,source,institution,citations_per_faculty,international_research_network,research_impact_score
1672,THE,Sapienza University of Rome,73.4,42.9,51.95
2642,THE,New Valley University,63.7,49.1,50.64
1237,QS,Instituto Tecnológico de Santo Domingo (INTEC),2.0,7.3,3.69
2374,THE,The Catholic University of America,40.5,48.6,38.20
1163,QS,University of Louisville,13.3,59.6,35.81
1422,QS,Harper Adams University,12.4,14.1,12.37
994,QS,University of Bahrain,5.0,38.7,21.06
2999,THE,Yokohama National University,22.1,35.3,20.51
1402,QS,Western Michigan University,10.7,33.0,21.06
1578,THE,University of North Carolina at Chapel Hill,91.3,49.9,65.56


In [17]:
# Module 3 - Global Ranking Score
# Uses each source's published 'overall_score' where available.
# Rows with no published overall_score (common for QS/THE band rows outside
# the top-ranked institutions) get an equivalent score derived from
# rank_numeric, min-max scaled to 0-100 within its own source
# (best rank = 100, worst rank = 0), so it sits on the same scale.

rank_based_score = pd.Series(np.nan, index=university_cleaned.index)
for s in ['QS', 'THE']:
    mask = src == s
    ranks = university_cleaned.loc[mask, 'rank_numeric']
    lo, hi = ranks.min(), ranks.max()
    rank_based_score.loc[mask] = 100 * (1 - (ranks - lo) / (hi - lo))

university_cleaned['global_ranking_score'] = (
    university_cleaned['overall_score'].fillna(rank_based_score).round(2)
)
university_cleaned[['source', 'institution', 'rank_numeric', 'overall_score', 'global_ranking_score']].sample(10, random_state=1)

,source,institution,rank_numeric,overall_score,global_ranking_score
1672,THE,Sapienza University of Rome,172.0,60.6510,60.65
2642,THE,New Valley University,1142.0,33.0210,33.02
1237,QS,Instituto Tecnológico de Santo Domingo (INTEC),1201.0,NaN,14.29
2374,THE,The Catholic University of America,874.0,37.6745,37.67
1163,QS,University of Louisville,1001.0,NaN,28.57
1422,QS,Harper Adams University,1401.0,NaN,0.00
994,QS,University of Bahrain,951.0,NaN,32.14
2999,THE,Yokohama National University,1499.0,27.2690,27.27
1402,QS,Western Michigan University,1201.0,NaN,14.29
1578,THE,University of North Carolina at Chapel Hill,78.0,70.2735,70.27


In [18]:
# Module 3 - Missing value check for new scores (evaluation target: 0% missing)
new_cols = ['global_ranking_score', 'research_impact_score', 'faculty_student_ratio_std']
print(university_cleaned[new_cols].isna().mean() * 100)

global_ranking_score         0.0
research_impact_score        0.0
faculty_student_ratio_std    0.0
dtype: float64


In [24]:
# Module 3 - Save deliverable
os.makedirs("data", exist_ok=True)
university_cleaned.to_excel("data/university_final_dataset.xlsx", index=False)
print("Saved: data/university_final_dataset.xlsx")
print("Final shape:", university_cleaned.shape)


Saved: data/university_final_dataset.xlsx
Final shape: (3692, 21)


In [25]:
# Module 3 - International Student Percentage
# NOTE: neither source publishes the raw enrollment percentage in this cleaned
# dataset — both only provide 'international_students_score', an index built
# from each body's own methodology. We min-max scale that index within each
# source to 0-100 so QS and THE institutions sit on the same comparable scale.
# Treat this as a standardized international-student index, not a literal
# headcount percentage.

international_student_percentage = pd.Series(np.nan, index=university_cleaned.index)
for s in ['QS', 'THE']:
    mask = src == s
    international_student_percentage.loc[mask] = minmax_0_100(
        university_cleaned.loc[mask, 'international_students_score']
    )
university_cleaned['international_student_percentage'] = international_student_percentage.round(2)
university_cleaned[['source', 'institution', 'international_students_score', 'international_student_percentage']].sample(10, random_state=1)

C:\Users\psaia\AppData\Local\Temp\ipykernel_18940\316869108.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[91.51515151515152 100.0 73.23232323232324 ... 2.0202020202020203
 3.5353535353535355 24.545454545454547]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  international_student_percentage.loc[mask] = minmax_0_100(


,source,institution,international_students_score,international_student_percentage
1672,THE,Sapienza University of Rome,7.0,7.291667
2642,THE,New Valley University,0.0,0.0
1237,QS,Instituto Tecnológico de Santo Domingo (INTEC),5.2,4.242424
2374,THE,The Catholic University of America,6.0,6.25
1163,QS,University of Louisville,5.9,4.949495
1422,QS,Harper Adams University,5.9,4.949495
994,QS,University of Bahrain,10.5,9.59596
2999,THE,Yokohama National University,8.0,8.333333
1402,QS,Western Michigan University,17.5,16.666667
1578,THE,University of North Carolina at Chapel Hill,10.0,10.416667


In [26]:
# Module 3 - Academic Reputation Score
# NOTE: QS publishes 'academic_reputation' (AR SCORE) directly. THE's cleaned
# schema does not include an equivalent reputation survey metric, so this
# column is genuinely unavailable (100% missing) for THE-sourced rows — it is
# NOT something we can safely fabricate from other columns without misleading
# results. We surface QS's own 0-100 score as-is (already a comparable scale)
# and leave THE rows as NaN.

academic_reputation_score = pd.Series(np.nan, index=university_cleaned.index)
qs_mask = src == 'QS'
academic_reputation_score.loc[qs_mask] = university_cleaned.loc[qs_mask, 'academic_reputation'].round(2)
university_cleaned['academic_reputation_score'] = academic_reputation_score

print("Academic Reputation Score - % missing by source:")
print(university_cleaned.groupby('source')['academic_reputation_score'].apply(lambda x: x.isna().mean() * 100))
university_cleaned[['source', 'institution', 'academic_reputation', 'academic_reputation_score']].sample(10, random_state=1)

Academic Reputation Score - % missing by source:
source
QS       0.0
THE    100.0
Name: academic_reputation_score, dtype: float64


,source,institution,academic_reputation,academic_reputation_score
1672,THE,Sapienza University of Rome,NaN,NaN
2642,THE,New Valley University,NaN,NaN
1237,QS,Instituto Tecnológico de Santo Domingo (INTEC),1.9,1.9
2374,THE,The Catholic University of America,NaN,NaN
1163,QS,University of Louisville,3.9,3.9
1422,QS,Harper Adams University,4.0,4.0
994,QS,University of Bahrain,14.3,14.3
2999,THE,Yokohama National University,NaN,NaN
1402,QS,Western Michigan University,5.0,5.0
1578,THE,University of North Carolina at Chapel Hill,NaN,NaN


In [27]:
# Module 3 - Research Productivity Index
# Standard proxy for research productivity is citations-per-faculty (research
# output relative to staff size), available for both sources. Min-max scaled
# within each source to 0-100 for comparability.

research_productivity_index = pd.Series(np.nan, index=university_cleaned.index)
for s in ['QS', 'THE']:
    mask = src == s
    research_productivity_index.loc[mask] = minmax_0_100(
        university_cleaned.loc[mask, 'citations_per_faculty']
    )
university_cleaned['research_productivity_index'] = research_productivity_index.round(2)
university_cleaned[['source', 'institution', 'citations_per_faculty', 'research_productivity_index']].sample(10, random_state=1)

,source,institution,citations_per_faculty,research_productivity_index
1672,THE,Sapienza University of Rome,73.4,72.59
2642,THE,New Valley University,63.7,62.45
1237,QS,Instituto Tecnológico de Santo Domingo (INTEC),2.0,1.01
2374,THE,The Catholic University of America,40.5,38.18
1163,QS,University of Louisville,13.3,12.42
1422,QS,Harper Adams University,12.4,11.52
994,QS,University of Bahrain,5.0,4.04
2999,THE,Yokohama National University,22.1,18.93
1402,QS,Western Michigan University,10.7,9.80
1578,THE,University of North Carolina at Chapel Hill,91.3,91.32


In [28]:
# Module 3 - Missing value check for the new metrics
new_cols = ['international_student_percentage', 'academic_reputation_score', 'research_productivity_index']
print(university_cleaned[new_cols].isna().mean() * 100)

international_student_percentage     1.056338
academic_reputation_score           59.344529
research_productivity_index          0.000000
dtype: float64


In [29]:
# Module 3 - Save deliverable
os.makedirs("data", exist_ok=True)
university_cleaned.to_excel("data/university_final_dataset.xlsx", index=False)
print("Saved: data/university_final_dataset.xlsx")
print("Final shape:", university_cleaned.shape)

Saved: data/university_final_dataset.xlsx
Final shape: (3692, 24)
